# Aging-clock checker v0.3
Compares published epigenetic clocks (via `pyaging`) with chronological age on three public datasets.

**Status:** this notebook was run top to bottom in a fresh Colab session on 2026-10-18 with no errors, and the summary table reproduced every number in `RESULTS.md` (earlier interactive runs, 2026-10-17). It is a demonstration on three datasets, not a benchmark; see the limits in `RESULTS.md`. Environment of that run: Python 3.13, pandas 3.0.6 (pip installed it for `pyaging` and Colab warned it conflicts with its pinned pandas 2.2.3; the run still worked).

**Rules:** 'error in years' is reported only for clocks trained to predict age (Horvath2013, AltumAge). GrimAge-type clocks (mortality) and DunedinPACE (pace) are shown by correlation and offset. Always check tissue first. Run datasets in order; each cell frees its big table.

**Memory:** run in a fresh session. If Colab restarts, re-run from the top.

In [ ]:
!pip -q install pyaging

In [ ]:
%%writefile clockcheck.py
"""clockcheck: compare epigenetic-clock outputs on one dataset.

Input: a table (one row per sample) with a chronological-age column and one
column per clock. Output: per-clock error in years, clock-vs-clock disagreement,
and plots. Pure pandas/numpy/matplotlib, so it can be tested without pyaging.

Clock kinds (needed because 'error in years' only makes sense for some):
  age      : trained to predict chronological age        -> error in years is valid
  mortality: trained on mortality risk, reported on an age scale -> NOT expected to equal
             chronological age; report the offset, do not call it an 'error'
  pace     : rate of aging (about 1.0 = average)         -> no years; compare by correlation only
These labels are from my reading of the clock papers and must be checked against
each clock's own metadata (adata.uns['<clock>_metadata']) before publishing.
"""
import numpy as np
import pandas as pd

CLOCK_KIND = {
    "horvath2013": "age",
    "altumage": "age",
    "hannum": "age",
    "pcgrimage": "mortality",
    "grimage2": "mortality",
    "phenoage": "mortality",
    "dunedinpace": "pace",
}


def kind_of(clock):
    return CLOCK_KIND.get(clock.lower(), "unknown")


def per_clock_stats(df, age_col, clocks):
    """Per-clock table. For kind=age: MAE, median AE, bias (mean pred-age), RMSE, Pearson r.
    For other kinds: only r and mean offset are reported and the error columns are NaN."""
    rows = []
    for c in clocks:
        d = df[[age_col, c]].dropna()
        n = len(d)
        k = kind_of(c)
        r = d[age_col].corr(d[c]) if n > 2 else np.nan
        row = {"clock": c, "kind": k, "n": n, "pearson_r": r,
               "mean_offset": (d[c] - d[age_col]).mean() if k != "pace" else np.nan}
        if k == "age":
            err = d[c] - d[age_col]
            row.update(MAE=err.abs().mean(), median_AE=err.abs().median(),
                       RMSE=float(np.sqrt((err ** 2).mean())), bias=err.mean())
        else:
            row.update(MAE=np.nan, median_AE=np.nan, RMSE=np.nan, bias=np.nan)
        rows.append(row)
    return pd.DataFrame(rows).set_index("clock")


def disagreement(df, clocks, age_col=None):
    """Pairwise mean absolute difference (years) between age-scale clocks, plus per-sample spread.
    Only clocks of kind 'age' are compared in years."""
    ac = [c for c in clocks if kind_of(c) == "age"]
    pair = pd.DataFrame(index=ac, columns=ac, dtype=float)
    for a in ac:
        for b in ac:
            if a == b:
                pair.loc[a, b] = 0.0
                continue
            d = df[[a, b]].dropna()
            pair.loc[a, b] = (d[a] - d[b]).abs().mean() if len(d) else np.nan
    spread = df[ac].max(axis=1) - df[ac].min(axis=1) if len(ac) > 1 else pd.Series(np.nan, index=df.index)
    return pair, spread.rename("max_minus_min_years")


def plot_all(df, age_col, clocks, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ac = [c for c in clocks if kind_of(c) == "age"]
    fig, axes = plt.subplots(1, max(len(ac), 1), figsize=(4.2 * max(len(ac), 1), 4), squeeze=False)
    for ax, c in zip(axes[0], ac):
        d = df[[age_col, c]].dropna()
        ax.scatter(d[age_col], d[c], s=10, alpha=.6)
        lo, hi = min(d[age_col].min(), d[c].min()), max(d[age_col].max(), d[c].max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        mae = (d[c] - d[age_col]).abs().mean()
        ax.set_title(f"{c}  (MAE {mae:.1f} y, n={len(d)})")
        ax.set_xlabel("chronological age (years)")
        ax.set_ylabel("predicted age (years)")
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


# ---------- GEO series-matrix loader (no GEOparse: reads the header directly, low memory) ----------
def load_geo_matrix(gse, keep=None, n=None, seed=0, workdir="."):
    """Download GSE<id>_series_matrix.txt.gz (if missing), parse sample characteristics from the
    header, optionally keep a subset of samples via keep(meta)->bool mask, optionally sample n at random,
    and return (df samples x probes as float32, meta). Fails loudly if the matrix has no data rows."""
    import gzip, os
    import numpy as np, pandas as pd
    fn = os.path.join(workdir, f"{gse}_series_matrix.txt.gz")
    if not os.path.exists(fn) or os.path.getsize(fn) < 1e5:
        url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{gse[:-3]}nnn/{gse}/matrix/{gse}_series_matrix.txt.gz"
        os.system(f"wget -q -O {fn} {url}")
    ids, chars = None, []
    with gzip.open(fn, "rt") as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"):
                ids = [x.strip('"') for x in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"):
                chars.append([x.strip('"') for x in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!series_matrix_table_begin"):
                break
    meta = pd.DataFrame(index=ids)
    for row in chars:
        key = row[0].split(": ", 1)[0]
        meta[key] = [v.split(": ", 1)[1] if ": " in v else v for v in row]
    sel = meta[keep(meta)] if keep is not None else meta
    if n is not None and n < len(sel):
        pick = list(np.random.default_rng(seed).choice(sel.index, size=n, replace=False))
    else:
        pick = list(sel.index)
    b = pd.read_csv(fn, sep="\t", comment="!", index_col=0, usecols=["ID_REF"] + pick,
                    compression="gzip", dtype={c: np.float32 for c in pick})
    if b.shape[0] == 0:
        raise ValueError(f"{gse}: series matrix has no data rows (raw IDAT only?)")
    return b.T, meta.loc[b.columns]


def run_clocks(df, drop_cols=("gender", "tissue_type", "dataset")):
    """Run the five clocks. PCGrimAge is run last in its own call (memory-heavy)."""
    import pyaging as pya
    meta_cols = [c for c in df.columns if c in drop_cols or c == "group"]
    ad = pya.pp.df_to_adata(df, metadata_cols=meta_cols, imputer_strategy="mean")
    pya.pred.predict_age(ad, ["Horvath2013", "AltumAge", "GrimAge2", "DunedinPACE"])
    pya.pred.predict_age(ad, ["PCGrimAge"])
    res = ad.obs.copy()
    res.columns = [c.lower() for c in res.columns]
    res["age"] = df["age"]
    return res


In [ ]:
import gc, numpy as np, pandas as pd, pyaging as pya
from clockcheck import *
clocks = ['horvath2013','altumage','pcgrimage','grimage2','dunedinpace']
results = {}

def report(name, res, subset=None):
    r = res if subset is None else res[subset]
    print('\n==', name, 'n =', len(r))
    display(per_clock_stats(r, 'age', clocks).round(2))
    pair, spread = disagreement(r, clocks); display(pair.round(2))
    for k in ['pcgrimage','grimage2']:
        off = r[k] - r['age']; s, i = np.polyfit(r['age'], off, 1)
        print(k, 'offset at age 30:', round(i+s*30,1), '| at 60:', round(i+s*60,1))

## A. Sperm (GSE139307, pyaging example): tissue-mismatch demonstration

In [ ]:
pya.data.download_example_data('GSE139307')
d = pd.read_pickle('pyaging_data/GSE139307.pkl')
d['female'] = (d['gender'] == 'F').astype(int)
results['A_sperm'] = run_clocks(d)
report('A: sperm GSE139307', results['A_sperm'])
del d; gc.collect()

## B. Whole blood (GSE41169, 450k): 33 controls + 62 schizophrenia
Caveat: dataset contact is Steve Horvath; possible training overlap, not checked. Controls are reported separately.

In [ ]:
d, meta = load_geo_matrix('GSE41169')
d['age'] = pd.to_numeric(meta['age'], errors='coerce').values
d['gender'] = meta['gender'].values
d['group'] = meta['diseasestatus (1=control, 2=scz patient)'].map({'1':'control','2':'scz'}).values
d['tissue_type'] = 'whole blood'; d['dataset'] = 'GSE41169'
d['female'] = (d['gender'].str.lower() == 'female').astype(int)
print(d.shape, d['group'].value_counts().to_dict())
results['B_blood'] = run_clocks(d)
results['B_blood']['group'] = d['group']
report('B: GSE41169 all', results['B_blood'])
report('B: GSE41169 controls', results['B_blood'], results['B_blood']['group']=='control')
del d; gc.collect()

## C. Peripheral blood leukocytes (GSE42861, 450k): 100 random healthy controls (label 'Normal')
Training overlap with any clock: not checked.

In [ ]:
d, meta = load_geo_matrix('GSE42861', keep=lambda m: m['subject']=='Normal', n=100, seed=0)
d['age'] = pd.to_numeric(meta['age'], errors='coerce').values
d['gender'] = meta['gender'].values
d['tissue_type'] = 'PBL'; d['dataset'] = 'GSE42861'
d['female'] = (d['gender'].str.lower().str[0] == 'f').astype(int)
print(d.shape, d['age'].describe().round(1).to_dict())
results['C_pbl'] = run_clocks(d)
report('C: GSE42861 controls', results['C_pbl'])
del d; gc.collect()

## Summary and checks
Check each clock's metadata (`adata.uns['<clock>_metadata']`) before quoting numbers; pyaging marks several as research-only and not yet author-approved.

In [ ]:
rows=[]
for name, r in results.items():
    sub = r[r['group']=='control'] if 'group' in r else r
    s = per_clock_stats(sub, 'age', clocks)
    rows.append(pd.concat({name: s[['n','pearson_r','mean_offset','MAE']]}))
display(pd.concat(rows).round(2))
plot_all(results['C_pbl'], 'age', clocks, 'clock_scatter_GSE42861.png')
from IPython.display import Image; Image('clock_scatter_GSE42861.png')

In [ ]:
import sys, importlib.metadata as md
print('python', sys.version.split()[0])
for p in ['pyaging','pandas','numpy','matplotlib']:
    try: print(p, md.version(p))
    except Exception as e: print(p, 'unknown')